In [2]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

from pyproj import Transformer
from sklearn.neighbors import BallTree
from pathlib import Path
import requests
import zipfile

In [3]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url_ubicacion_dataset = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico"

response = requests.get(url_ubicacion_dataset)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

enlaces = []

for a in soup.find_all("a", href=True):
    href = urljoin(url_ubicacion_dataset, a["href"])
    texto = a.get_text(" ", strip=True)

    if "/download/" in href or href.lower().endswith((".csv", ".xlsx", ".zip")):
        enlaces.append({
            "texto": texto,
            "url": href
        })

df_enlaces = pd.DataFrame(enlaces).drop_duplicates()
df_enlaces.head(20)

,texto,url
0,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
1,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
2,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
3,Descarga,https://datos.madrid.es/dataset/202468-0-inten...


In [7]:
df_enlaces[df_enlaces["url"].str.contains(".csv", case=False, na=False)]

,texto,url


In [9]:
url_csv = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico/resource/202468-294-intensidad-trafico/download/202468-294-intensidad-trafico.csv"

df_medidores = pd.read_csv(
    url_csv,
    sep=";",
    encoding="latin1"
)

df_medidores.head()

,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,other,1.0,6835,18RA28PM01,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,other,9.0,1012,18RA66PM01,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,URB,10.0,5035,95013,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,URB,5.0,5579,61068,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,URB,5.0,5580,61069,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [10]:
df_medidores.columns

Index(['tipo_elem', 'distrito', 'id', 'cod_cent', 'nombre', 'utm_x', 'utm_y',
       'longitud', 'latitud'],
      dtype='object')

In [12]:
medidores = df_medidores[["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [13]:
print("Número de medidores:", len(medidores))
medidores.head()

Número de medidores: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [14]:
medidores.dtypes

id            int64
nombre       object
utm_x       float64
utm_y       float64
longitud    float64
latitud     float64
dtype: object

In [15]:
medidores[["latitud", "longitud"]].describe()

,latitud,longitud
count,5072.000000,5072.000000
mean,40.430447,-3.684001
std,0.039163,0.042728
min,40.332454,-3.836886
25%,40.398978,-3.712553
50%,40.431302,-3.686923
75%,40.460080,-3.656194
max,40.515611,-3.551623


In [16]:
G_osm = ox.graph_from_place(
    "Madrid, Spain",
    network_type="drive",
    simplify=True
)

print("Nodos OSM:", len(G_osm.nodes))
print("Aristas OSM:", len(G_osm.edges))

Nodos OSM: 31453
Aristas OSM: 61857


In [17]:
# Comprobar IDs duplicados
print("IDs duplicados:", medidores["id"].duplicated().sum())

# Comprobar coordenadas nulas
print("Latitud nula:", medidores["latitud"].isna().sum())
print("Longitud nula:", medidores["longitud"].isna().sum())

# Comprobar coordenadas fuera de un rango razonable para Madrid
fuera_madrid = medidores[
    ~(
        (medidores["latitud"].between(40.30, 40.55)) &
        (medidores["longitud"].between(-3.90, -3.50))
    )
]

print("Medidores fuera de rango Madrid:", len(fuera_madrid))
fuera_madrid.head()

IDs duplicados: 0
Latitud nula: 0
Longitud nula: 0
Medidores fuera de rango Madrid: 0


,id,nombre,utm_x,utm_y,longitud,latitud


In [18]:
osm_nodes, distancias = ox.distance.nearest_nodes(
    G_osm,
    X=medidores["longitud"].values,
    Y=medidores["latitud"].values,
    return_dist=True
)

medidores["osm_node"] = osm_nodes
medidores["distancia_osm_node_m"] = distancias

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,32636471,79.621034
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,315259372,54.829432
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,305399713,15.640852
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,1672792326,40.116809
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,119794656,14.979371


In [19]:
medidores["distancia_osm_node_m"].describe()

count    5072.000000
mean       36.635675
std        28.015551
min         0.101869
25%        16.227750
50%        28.117214
75%        50.402014
max       345.242185
Name: distancia_osm_node_m, dtype: float64

In [20]:
medidores.sort_values("distancia_osm_node_m", ascending=False).head(20)

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,434514.277528,4.468632e+06,-3.771300,40.365688,306400716,345.242185
107,4928,(TACTICO) AV. POBLADOS O-E (GIRO A ERICA),435668.604184,4.470451e+06,-3.757889,40.382164,282940163,228.695575
1394,4960,(TACTICO) ERICA N-S (CENTRO C.I.E.),435691.441140,4.470458e+06,-3.757621,40.382233,282940163,219.565346
1760,11199,Fuerzas Armadas - Ciudad Deportiva O-E - Fuerz...,448191.892131,4.481464e+06,-3.611259,40.482247,1012899036,178.720804
2325,11200,Fuerzas Armadas - Ciudad Deportiva O-E (Vía Se...,448192.910496,4.481435e+06,-3.611244,40.481993,1012899118,178.333987
4888,6876,12XC06PM01,441861.911399,4.471142e+06,-3.684994,40.388851,317771984,169.480943
1745,11191,"Av Fuerzas Armadas, 322 O-E - Av Fuerzas Armad...",447371.205461,4.481464e+06,-3.620941,40.482198,969169634,168.795375
1759,11192,"Av Fuerzas Armadas, 322 O-E (Via Servicio) - A...",447372.223825,4.481436e+06,-3.620927,40.481944,969169634,168.326941
446,9916,SINESIO DELGADO O-E (HOSPITAL CARLOS III-ENTRA...,440954.801598,4.480675e+06,-3.696566,40.474658,26205041,163.715082
445,9915,SINESIO DELGADO E-O (SALIDA TUNEL-HOSPITAL CAR...,440949.748176,4.480680e+06,-3.696627,40.474708,26205041,156.229910


In [21]:
import numpy as np
from sklearn.neighbors import BallTree

# Coordenadas en radianes para distancia haversine
coords = np.radians(medidores[["latitud", "longitud"]].values)

tree = BallTree(coords, metric="haversine")

# Calculamos hasta los 20 vecinos más cercanos para estudiar la distribución
K_ANALISIS = 20

distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000  # radio medio de la Tierra en metros

# Convertimos distancias a metros
distancias_m = distancias * R

In [22]:
resumen_vecinos = pd.DataFrame({
    "vecino_1_m": distancias_m[:, 1],
    "vecino_3_m": distancias_m[:, 3],
    "vecino_5_m": distancias_m[:, 5],
    "vecino_10_m": distancias_m[:, 10],
    "vecino_20_m": distancias_m[:, 20],
})

resumen_vecinos.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

,vecino_1_m,vecino_3_m,vecino_5_m,vecino_10_m,vecino_20_m
count,5072.000000,5072.000000,5072.000000,5072.000000,5072.000000
mean,60.299512,134.802462,192.218004,303.847693,479.846961
std,50.277204,83.244571,106.243956,150.109013,302.088483
min,0.000000,10.896282,15.656927,80.867585,195.699433
25%,16.317860,89.649261,130.094414,216.112762,344.834643
50%,51.730719,125.062705,172.480334,274.661836,421.074841
75%,90.709754,164.091985,225.365385,348.903363,527.182705
90%,123.780820,211.121938,299.572190,449.004668,660.410756
95%,148.631821,252.537244,367.262305,544.203109,817.526027
max,559.979456,1626.513574,1656.200417,1741.671511,4377.116647


In [23]:
radio_candidatos_m = np.percentile(distancias_m[:, 5], 90)

print("Radio de candidatos calculado:", radio_candidatos_m, "metros")

Radio de candidatos calculado: 299.5721899933185 metros


In [24]:
# Convertimos radio de metros a radianes
radio_candidatos_rad = radio_candidatos_m / R

indices_radio, distancias_radio = tree.query_radius(
    coords,
    r=radio_candidatos_rad,
    return_distance=True,
    sort_results=True
)

pares_candidatos = []

for i in range(len(medidores)):
    medidor_origen = medidores.iloc[i]
    
    for j, distancia_rad in zip(indices_radio[i], distancias_radio[i]):
        # Saltamos el propio medidor
        if i == j:
            continue
        
        medidor_destino = medidores.iloc[j]
        
        pares_candidatos.append({
            "id_origen": medidor_origen["id"],
            "id_destino": medidor_destino["id"],
            "distancia_directa_m": distancia_rad * R,
            "osm_node_origen": medidor_origen["osm_node"],
            "osm_node_destino": medidor_destino["osm_node"]
        })

df_pares = pd.DataFrame(pares_candidatos)

print("Número de pares candidatos:", len(df_pares))
df_pares.head()

Número de pares candidatos: 60494


,id_origen,id_destino,distancia_directa_m,osm_node_origen,osm_node_destino
0,6835,6833,20.104431,32636471,32636471
1,6835,6836,91.872471,32636471,315261895
2,6835,6837,93.934596,32636471,315264896
3,6835,6827,110.969150,32636471,315261895
4,6835,1042,116.845927,32636471,315265031


In [27]:
candidatos_por_medidor = (
    df_pares
    .groupby("id_origen")
    .size()
    .reset_index(name="num_candidatos")
)

candidatos_por_medidor["num_candidatos"].describe()


count    5060.000000
mean       11.955336
std         6.228532
min         1.000000
25%         7.000000
50%        11.000000
75%        16.000000
max        38.000000
Name: num_candidatos, dtype: float64

In [28]:
candidatos_por_medidor.sort_values("num_candidatos").head(20)

,id_origen,num_candidatos
3186,6768,1
3458,7122,1
331,3706,1
3121,6697,1
3117,6693,1
3113,6689,1
3116,6692,1
1498,4988,1
2792,6352,1
2575,6125,1


In [29]:
# Diccionario: nodo OSM -> lista de medidores asociados a ese nodo
osm_node_to_medidores = (
    medidores
    .groupby("osm_node")["id"]
    .apply(list)
    .to_dict()
)

# Grafo final de medidores
G_medidores = nx.DiGraph()

# Añadimos todos los medidores como nodos, sin eliminar ninguno
for _, row in medidores.iterrows():
    G_medidores.add_node(
        row["id"],
        nombre=row["nombre"],
        latitud=row["latitud"],
        longitud=row["longitud"],
        osm_node=row["osm_node"]
    )

print("Nodos en grafo de medidores:", G_medidores.number_of_nodes())

Nodos en grafo de medidores: 5072


In [30]:
def construir_aristas_por_cutoff(medidores, G_osm, cutoff_m):
    osm_node_to_medidores = (
        medidores
        .groupby("osm_node")["id"]
        .apply(list)
        .to_dict()
    )

    aristas = []

    for _, row in medidores.iterrows():
        id_origen = row["id"]
        osm_origen = row["osm_node"]

        try:
            distancias_red = nx.single_source_dijkstra_path_length(
                G_osm,
                source=osm_origen,
                cutoff=cutoff_m,
                weight="length"
            )
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

        for osm_destino, distancia_red_m in distancias_red.items():
            if osm_destino not in osm_node_to_medidores:
                continue

            for id_destino in osm_node_to_medidores[osm_destino]:
                if id_destino == id_origen:
                    continue

                aristas.append({
                    "id_origen": id_origen,
                    "id_destino": id_destino,
                    "osm_node_origen": osm_origen,
                    "osm_node_destino": osm_destino,
                    "distancia_red_m": distancia_red_m
                })

    return pd.DataFrame(aristas).drop_duplicates()

In [3]:
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx

from sklearn.neighbors import BallTree
from tqdm import tqdm

# URL del dataset de ubicación de puntos medidores
url_csv = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico/resource/202468-294-intensidad-trafico/download/202468-294-intensidad-trafico.csv"

df_medidores = pd.read_csv(
    url_csv,
    sep=";",
    encoding="latin1"
)

# Crear tabla limpia de medidores
medidores = df_medidores[
    ["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]
].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

# Asegurar que latitud y longitud son numéricas
medidores["latitud"] = (
    medidores["latitud"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

medidores["longitud"] = (
    medidores["longitud"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print("Medidores cargados:", len(medidores))
medidores.head()

Medidores cargados: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [5]:
G_osm = ox.graph_from_place(
    "Madrid, Spain",
    network_type="drive",
    simplify=True
)

print("Nodos OSM:", len(G_osm.nodes))
print("Aristas OSM:", len(G_osm.edges))

Nodos OSM: 31453
Aristas OSM: 61857


In [6]:
osm_nodes, distancias = ox.distance.nearest_nodes(
    G_osm,
    X=medidores["longitud"].values,
    Y=medidores["latitud"].values,
    return_dist=True
)

medidores["osm_node"] = osm_nodes
medidores["distancia_osm_node_m"] = distancias

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315,32636471,79.621034
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861,315259372,54.829432
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040,305399713,15.640852
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932,1672792326,40.116809
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073,119794656,14.979371


In [9]:
coords = np.radians(medidores[["latitud", "longitud"]].values)

tree = BallTree(coords, metric="haversine")

K_ANALISIS = 20
distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000
distancias_m = distancias * R

radio_candidatos_m = np.percentile(distancias_m[:, 5], 90)

print("Radio de candidatos calculado:", radio_candidatos_m, "metros")

radio_candidatos_rad = radio_candidatos_m / R

indices_radio, distancias_radio = tree.query_radius(
    coords,
    r=radio_candidatos_rad,
    return_distance=True,
    sort_results=True
)

pares_candidatos = []

for i in range(len(medidores)):
    medidor_origen = medidores.iloc[i]
    
    for j, distancia_rad in zip(indices_radio[i], distancias_radio[i]):
        if i == j:
            continue
        
        medidor_destino = medidores.iloc[j]
        
        pares_candidatos.append({
            "id_origen": medidor_origen["id"],
            "id_destino": medidor_destino["id"],
            "distancia_directa_m": distancia_rad * R,
            "osm_node_origen": medidor_origen["osm_node"],
            "osm_node_destino": medidor_destino["osm_node"]
        })

df_pares = pd.DataFrame(pares_candidatos).drop_duplicates()

print("Número de pares candidatos:", len(df_pares))
df_pares.head()

Radio de candidatos calculado: 299.5721899933185 metros
Número de pares candidatos: 60494


,id_origen,id_destino,distancia_directa_m,osm_node_origen,osm_node_destino
0,6835,6833,20.104431,32636471,32636471
1,6835,6836,91.872471,32636471,315261895
2,6835,6837,93.934596,32636471,315264896
3,6835,6827,110.969150,32636471,315261895
4,6835,1042,116.845927,32636471,315265031


In [10]:
import numpy as np
from sklearn.neighbors import BallTree

# Coordenadas de medidores en radianes
coords = np.radians(medidores[["latitud", "longitud"]].values)

# Árbol espacial para buscar vecinos cercanos
tree = BallTree(coords, metric="haversine")

# Analizamos vecinos cercanos para obtener un radio basado en datos reales
K_ANALISIS = 20
distancias, indices = tree.query(coords, k=K_ANALISIS + 1)

R = 6371000  # radio medio de la Tierra en metros
distancias_m = distancias * R

# Resumen de distancias a distintos vecinos
resumen_vecinos = pd.DataFrame({
    "vecino_1_m": distancias_m[:, 1],
    "vecino_3_m": distancias_m[:, 3],
    "vecino_5_m": distancias_m[:, 5],
    "vecino_10_m": distancias_m[:, 10],
    "vecino_20_m": distancias_m[:, 20],
})

resumen_vecinos.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95])

,vecino_1_m,vecino_3_m,vecino_5_m,vecino_10_m,vecino_20_m
count,5072.000000,5072.000000,5072.000000,5072.000000,5072.000000
mean,60.299512,134.802462,192.218004,303.847693,479.846961
std,50.277204,83.244571,106.243956,150.109013,302.088483
min,0.000000,10.896282,15.656927,80.867585,195.699433
25%,16.317860,89.649261,130.094414,216.112762,344.834643
50%,51.730719,125.062705,172.480334,274.661836,421.074841
75%,90.709754,164.091985,225.365385,348.903363,527.182705
90%,123.780820,211.121938,299.572190,449.004668,660.410756
95%,148.631821,252.537244,367.262305,544.203109,817.526027
max,559.979456,1626.513574,1656.200417,1741.671511,4377.116647


In [11]:
from tqdm import tqdm

aristas_reales = []

for _, row in tqdm(df_pares.iterrows(), total=len(df_pares)):
    id_origen = row["id_origen"]
    id_destino = row["id_destino"]
    
    osm_origen = row["osm_node_origen"]
    osm_destino = row["osm_node_destino"]
    
    distancia_directa_m = row["distancia_directa_m"]
    
    # Caso especial: dos medidores asociados al mismo nodo OSM
    if osm_origen == osm_destino:
        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": 0,
            "tipo_conexion": "mismo_nodo_osm"
        })
        continue
    
    try:
        distancia_red_m = nx.shortest_path_length(
            G_osm,
            source=osm_origen,
            target=osm_destino,
            weight="length"
        )
        
        aristas_reales.append({
            "id_origen": id_origen,
            "id_destino": id_destino,
            "osm_node_origen": osm_origen,
            "osm_node_destino": osm_destino,
            "distancia_directa_m": distancia_directa_m,
            "distancia_red_m": distancia_red_m,
            "tipo_conexion": "camino_osm"
        })
        
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        # Si no hay camino real en OSM, no se crea arista
        continue

df_aristas_reales = pd.DataFrame(aristas_reales)

print("Pares candidatos:", len(df_pares))
print("Aristas reales encontradas:", len(df_aristas_reales))

df_aristas_reales.head()

100%|██████████| 60494/60494 [01:26<00:00, 700.60it/s] 


Pares candidatos: 60494
Aristas reales encontradas: 60251


,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion
0,6835.0,6833.0,32636471.0,32636471.0,20.104431,0.000000,mismo_nodo_osm
1,6835.0,6836.0,32636471.0,315261895.0,91.872471,6655.017406,camino_osm
2,6835.0,6837.0,32636471.0,315264896.0,93.934596,2248.920678,camino_osm
3,6835.0,6827.0,32636471.0,315261895.0,110.969150,6655.017406,camino_osm
4,6835.0,1042.0,32636471.0,315265031.0,116.845927,2583.076553,camino_osm


In [12]:
df_aristas_reales["factor_rodeo"] = (
    df_aristas_reales["distancia_red_m"] /
    df_aristas_reales["distancia_directa_m"].replace(0, np.nan)
)

df_aristas_reales[[
    "distancia_directa_m",
    "distancia_red_m",
    "factor_rodeo"
]].describe()

,distancia_directa_m,distancia_red_m,factor_rodeo
count,60251.000000,60251.000000,60247.000000
mean,182.030328,634.300037,4.189533
std,77.679277,1101.004101,12.548453
min,0.000000,0.000000,0.000000
25%,124.633069,190.874185,1.098552
50%,190.990925,332.662946,1.647146
75%,248.275283,588.988119,3.171776
max,299.571214,13963.539403,734.950412


In [13]:
df_aristas_reales.sort_values("factor_rodeo", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo
57566,6952.0,6949.0,5.360985e+09,3.079971e+08,12.250197,9003.287558,camino_osm,734.950412
46633,1015.0,1016.0,3.389200e+08,2.172323e+07,8.791932,6123.576228,camino_osm,696.499499
9513,11370.0,6786.0,2.554991e+07,2.977675e+08,13.582513,6412.259469,camino_osm,472.096692
57567,6952.0,6950.0,5.360985e+09,3.079971e+08,19.560019,9003.287558,camino_osm,460.290317
9540,1052.0,1049.0,3.880871e+08,2.493682e+09,25.653528,11766.287184,camino_osm,458.661561
57569,6951.0,6949.0,5.360985e+09,3.079971e+08,19.911916,9003.287558,camino_osm,452.155757
1162,11373.0,11374.0,9.823065e+09,4.294615e+08,18.481650,6765.444483,camino_osm,366.062801
56893,3826.0,6790.0,2.152588e+07,2.095326e+07,6.961135,2341.325133,camino_osm,336.342442
1163,11373.0,11375.0,9.823065e+09,4.294615e+08,21.174514,6765.444483,camino_osm,319.508849
26829,6738.0,6737.0,2.593885e+07,2.537145e+09,18.511306,5833.573622,camino_osm,315.135710


In [14]:
df_aristas_reales["calidad_arista"] = "ok"

df_aristas_reales.loc[
    df_aristas_reales["factor_rodeo"] > 10,
    "calidad_arista"
] = "revisar_rodeo_alto"

df_aristas_reales.loc[
    df_aristas_reales["distancia_red_m"] > 2000,
    "calidad_arista"
] = "revisar_distancia_red_alta"

df_aristas_reales["calidad_arista"].value_counts()

calidad_arista
ok                            55211
revisar_distancia_red_alta     3957
revisar_rodeo_alto             1083
Name: count, dtype: int64

In [15]:
G_medidores = nx.DiGraph()

for _, row in medidores.iterrows():
    G_medidores.add_node(
        row["id"],
        nombre=row["nombre"],
        latitud=row["latitud"],
        longitud=row["longitud"],
        osm_node=row["osm_node"]
    )

for _, row in df_aristas_reales.iterrows():
    G_medidores.add_edge(
        row["id_origen"],
        row["id_destino"],
        distancia_directa_m=row["distancia_directa_m"],
        distancia_red_m=row["distancia_red_m"],
        factor_rodeo=row["factor_rodeo"],
        weight=row["distancia_red_m"],
        tipo_conexion=row["tipo_conexion"]
    )

print("Nodos:", G_medidores.number_of_nodes())
print("Aristas:", G_medidores.number_of_edges())

Nodos: 5072
Aristas: 60251


In [16]:
nodos_con_aristas = set(df_aristas_reales["id_origen"]).union(
    set(df_aristas_reales["id_destino"])
)

print("Medidores totales:", medidores["id"].nunique())
print("Medidores con alguna arista:", len(nodos_con_aristas))
print("Medidores aislados:", medidores["id"].nunique() - len(nodos_con_aristas))

print("Componentes débiles:", nx.number_weakly_connected_components(G_medidores))
print("Componentes fuertes:", nx.number_strongly_connected_components(G_medidores))

Medidores totales: 5072
Medidores con alguna arista: 5060
Medidores aislados: 12
Componentes débiles: 79
Componentes fuertes: 98


## 12 NODOS AISLADOS: REVISIÓN

In [17]:
ids_aislados = set(medidores["id"]) - nodos_con_aristas

medidores_aislados = medidores[
    medidores["id"].isin(ids_aislados)
].copy()

medidores_aislados

,id,nombre,utm_x,utm_y,longitud,latitud,osm_node,distancia_osm_node_m
345,10210,PM43041,441959.264428,4.485213e+06,-3.685133,40.515611,2590772533,101.383315
675,5298,(TACTICO)ALLARIZ O-E(PROGRESO-AV. CARABANCHEL ...,436016.935921,4.468817e+06,-3.753622,40.367476,306163498,16.163931
872,6923,San Cipriano - Efigencia-Caños San Pedro,448977.298430,4.472756e+06,-3.601293,40.403851,307534056,20.827393
1938,5268,(TACTICO)SALIDA POLIGONO N-S,434514.277528,4.468632e+06,-3.771300,40.365688,306400716,345.242185
2302,6489,Embajadores - Santa Catalina-Carretera Villave...,442608.127625,4.468936e+06,-3.676004,40.369021,306101165,90.843345
2761,5160,(TACTICO)JOSE CADALSO S-N(VALLE INCLAN-AV. LAS...,434473.153008,4.470500e+06,-3.771977,40.382518,26085559,35.193999
3214,4868,(TACTICO) BATALLA GARELLANO Nº 27 S-N (SIRRACH...,432749.895389,4.478580e+06,-3.793131,40.455164,292702537,15.418926
3540,10012,(TACTICO) Salida Clinica Lopez Ibor,438594.297374,4.479950e+06,-3.724342,40.467961,4777894121,13.405010
4237,5803,Martínez de la Riva - Arroyo del Olivar-Párroc...,443471.998729,4.471604e+06,-3.666066,40.393116,247989374,9.468668
4388,6584,(TACTICO) SALIDA CUARTEL ARTILLERIA,442462.065684,4.484965e+06,-3.679176,40.513409,255961297,90.867682


In [18]:
medidores_aislados[[
    "id",
    "nombre",
    "latitud",
    "longitud",
    "osm_node",
    "distancia_osm_node_m"
]].sort_values("distancia_osm_node_m", ascending=False)

,id,nombre,latitud,longitud,osm_node,distancia_osm_node_m
1938,5268,(TACTICO)SALIDA POLIGONO N-S,40.365688,-3.771300,306400716,345.242185
345,10210,PM43041,40.515611,-3.685133,2590772533,101.383315
4388,6584,(TACTICO) SALIDA CUARTEL ARTILLERIA,40.513409,-3.679176,255961297,90.867682
2302,6489,Embajadores - Santa Catalina-Carretera Villave...,40.369021,-3.676004,306101165,90.843345
2761,5160,(TACTICO)JOSE CADALSO S-N(VALLE INCLAN-AV. LAS...,40.382518,-3.771977,26085559,35.193999
872,6923,San Cipriano - Efigencia-Caños San Pedro,40.403851,-3.601293,307534056,20.827393
4743,3528,Tumaco - Tumaco-Tampico,40.445069,-3.635408,114123521,20.281911
675,5298,(TACTICO)ALLARIZ O-E(PROGRESO-AV. CARABANCHEL ...,40.367476,-3.753622,306163498,16.163931
3214,4868,(TACTICO) BATALLA GARELLANO Nº 27 S-N (SIRRACH...,40.455164,-3.793131,292702537,15.418926
3540,10012,(TACTICO) Salida Clinica Lopez Ibor,40.467961,-3.724342,4777894121,13.405010


In [19]:
aristas_revision = df_aristas_reales[
    df_aristas_reales["calidad_arista"] != "ok"
].copy()

aristas_revision.head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,calidad_arista
1,6835.0,6836.0,32636471.0,3.152619e+08,91.872471,6655.017406,camino_osm,72.437557,revisar_distancia_red_alta
2,6835.0,6837.0,32636471.0,3.152649e+08,93.934596,2248.920678,camino_osm,23.941346,revisar_distancia_red_alta
3,6835.0,6827.0,32636471.0,3.152619e+08,110.969150,6655.017406,camino_osm,59.971779,revisar_distancia_red_alta
4,6835.0,1042.0,32636471.0,3.152650e+08,116.845927,2583.076553,camino_osm,22.106689,revisar_distancia_red_alta
5,6835.0,6838.0,32636471.0,3.152649e+08,117.342223,2248.920678,camino_osm,19.165486,revisar_distancia_red_alta
15,6835.0,6839.0,32636471.0,2.172329e+07,241.586247,4441.731134,camino_osm,18.385695,revisar_distancia_red_alta
18,6835.0,1012.0,32636471.0,3.152594e+08,284.336468,5450.813054,camino_osm,19.170292,revisar_distancia_red_alta
20,6835.0,1014.0,32636471.0,3.152594e+08,295.894752,5450.813054,camino_osm,18.421459,revisar_distancia_red_alta
24,1012.0,1016.0,315259372.0,2.172323e+07,66.215512,4972.088987,camino_osm,75.089490,revisar_distancia_red_alta
25,1012.0,1015.0,315259372.0,3.389200e+08,72.877527,3027.126341,camino_osm,41.537172,revisar_distancia_red_alta


In [20]:
aristas_revision[[
    "distancia_directa_m",
    "distancia_red_m",
    "factor_rodeo",
    "calidad_arista"
]].describe()

,distancia_directa_m,distancia_red_m,factor_rodeo
count,5040.000000,5040.000000,5040.000000
mean,168.099788,3344.425904,27.402905
std,83.588720,2304.184072,35.535166
min,2.780161,39.721970,6.780740
25%,101.616453,2095.837880,12.021858
50%,171.307070,2839.105725,16.331936
75%,242.614543,3992.333066,27.459901
max,299.492500,13963.539403,734.950412


In [21]:
df_aristas_reales.sort_values("factor_rodeo", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,calidad_arista
57566,6952.0,6949.0,5.360985e+09,3.079971e+08,12.250197,9003.287558,camino_osm,734.950412,revisar_distancia_red_alta
46633,1015.0,1016.0,3.389200e+08,2.172323e+07,8.791932,6123.576228,camino_osm,696.499499,revisar_distancia_red_alta
9513,11370.0,6786.0,2.554991e+07,2.977675e+08,13.582513,6412.259469,camino_osm,472.096692,revisar_distancia_red_alta
57567,6952.0,6950.0,5.360985e+09,3.079971e+08,19.560019,9003.287558,camino_osm,460.290317,revisar_distancia_red_alta
9540,1052.0,1049.0,3.880871e+08,2.493682e+09,25.653528,11766.287184,camino_osm,458.661561,revisar_distancia_red_alta
57569,6951.0,6949.0,5.360985e+09,3.079971e+08,19.911916,9003.287558,camino_osm,452.155757,revisar_distancia_red_alta
1162,11373.0,11374.0,9.823065e+09,4.294615e+08,18.481650,6765.444483,camino_osm,366.062801,revisar_distancia_red_alta
56893,3826.0,6790.0,2.152588e+07,2.095326e+07,6.961135,2341.325133,camino_osm,336.342442,revisar_distancia_red_alta
1163,11373.0,11375.0,9.823065e+09,4.294615e+08,21.174514,6765.444483,camino_osm,319.508849,revisar_distancia_red_alta
26829,6738.0,6737.0,2.593885e+07,2.537145e+09,18.511306,5833.573622,camino_osm,315.135710,revisar_distancia_red_alta


In [22]:
df_aristas_reales.sort_values("distancia_red_m", ascending=False).head(20)

,id_origen,id_destino,osm_node_origen,osm_node_destino,distancia_directa_m,distancia_red_m,tipo_conexion,factor_rodeo,calidad_arista
10268,11496.0,6856.0,3.152449e+08,2.136385e+09,218.721078,13963.539403,camino_osm,63.841764,revisar_distancia_red_alta
10262,11496.0,6858.0,3.152449e+08,2.136385e+09,209.732125,13963.539403,camino_osm,66.577971,revisar_distancia_red_alta
10259,11496.0,6857.0,3.152449e+08,2.136385e+09,208.098671,13963.539403,camino_osm,67.100570,revisar_distancia_red_alta
10272,11496.0,7118.0,3.152449e+08,2.136385e+09,239.828567,13963.539403,camino_osm,58.223003,revisar_distancia_red_alta
10270,11496.0,1032.0,3.152449e+08,2.136385e+09,231.140447,13963.539403,camino_osm,60.411493,revisar_distancia_red_alta
10269,11496.0,7145.0,3.152449e+08,2.136385e+09,219.215723,13963.539403,camino_osm,63.697709,revisar_distancia_red_alta
49876,1035.0,6857.0,3.152449e+08,2.136385e+09,204.252481,13963.539403,camino_osm,68.364112,revisar_distancia_red_alta
49877,1035.0,6858.0,3.152449e+08,2.136385e+09,206.291569,13963.539403,camino_osm,67.688367,revisar_distancia_red_alta
49881,1035.0,6856.0,3.152449e+08,2.136385e+09,214.867578,13963.539403,camino_osm,64.986721,revisar_distancia_red_alta
49888,1035.0,7118.0,3.152449e+08,2.136385e+09,237.149216,13963.539403,camino_osm,58.880816,revisar_distancia_red_alta


In [23]:
df_aristas_reales["distancia_red_m"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
)

count    60251.000000
mean       634.300037
std       1101.004101
min          0.000000
50%        332.662946
75%        588.988119
90%       1315.042019
95%       2565.392329
99%       5247.698913
max      13963.539403
Name: distancia_red_m, dtype: float64

In [24]:
df_aristas_reales["factor_rodeo"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
)

count    60247.000000
mean         4.189533
std         12.548453
min          0.000000
50%          1.647146
75%          3.171776
90%          7.481676
95%         14.443565
99%         47.567329
max        734.950412
Name: factor_rodeo, dtype: float64

In [25]:
umbral_distancia_red = df_aristas_reales["distancia_red_m"].quantile(0.95)
umbral_rodeo = df_aristas_reales["factor_rodeo"].quantile(0.95)

print("Umbral distancia red P95:", umbral_distancia_red)
print("Umbral rodeo P95:", umbral_rodeo)

Umbral distancia red P95: 2565.3923293443236
Umbral rodeo P95: 14.443565266456861


In [26]:
df_aristas_reales["calidad_arista"] = "ok"

df_aristas_reales.loc[
    df_aristas_reales["distancia_red_m"] > umbral_distancia_red,
    "calidad_arista"
] = "revisar_distancia_red_alta"

df_aristas_reales.loc[
    df_aristas_reales["factor_rodeo"] > umbral_rodeo,
    "calidad_arista"
] = "revisar_rodeo_alto"

df_aristas_reales["calidad_arista"].value_counts()

calidad_arista
ok                            56484
revisar_rodeo_alto             3013
revisar_distancia_red_alta      754
Name: count, dtype: int64